# Pengambilan Data

Pengambilan **data riil dari satelit Sentinel-5P TROPOMI** melalui Copernicus Data Space Ecosystem (CDSE) openEO API secara bertahap (satu polutan per sel) dan menyimpannya ke dalam **1 file CSV tunggal** di folder `data/data_kualitas_udara_mendenrejo.csv`.

### Satuan Nilai Asli Satelit Sentinel-5P L2:
- **`CO`** (Carbon Monoxide): $\text{mol/m}^2$ (Kolom total atmosfer, rentang riil $\approx 0.01 - 0.05\,\text{mol/m}^2$)
- **`NO2`** (Nitrogen Dioxide): $\text{mol/m}^2$ (Kolom troposferik, rentang riil $\approx 0.00001 - 0.0001\,\text{mol/m}^2$)
- **`SO2`** (Sulfur Dioxide): $\text{mol/m}^2$ (Kolom total atmosfer, rentang riil $\approx -0.0001 - 0.001\,\text{mol/m}^2$)
- **`CH4`** (Methane): $\text{ppb}$ / $\text{mol/m}^2$

*(Catatan: Hari dengan tutupan awan tebal / tanpa lintasan satelit akan bernilai `NaN`/kosong secara alami dari data satelit)*

--- 
### Sel 1: Setup Lingkungan, Autentikasi CDSE Copernicus & Inisialisasi Kerangka CSV

In [3]:
import os
import json
import pandas as pd
import numpy as np
import openeo
import logging

# 1. Penentuan Folder
cwd = os.path.abspath(os.getcwd())
if cwd.endswith('2-pengambilan_data'):
    EXP_DIR = cwd
elif os.path.exists(os.path.join(cwd, 'src', '2-pengambilan_data')):
    EXP_DIR = os.path.join(cwd, 'src', '2-pengambilan_data')
elif os.path.exists(os.path.join(cwd, '2-pengambilan_data')):
    EXP_DIR = os.path.join(cwd, '2-pengambilan_data')
else:
    EXP_DIR = cwd

env_file = os.path.join(EXP_DIR, '.env')
DATA_DIR = os.path.join(EXP_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)
csv_file = os.path.join(DATA_DIR, 'data_kualitas_udara_mendenrejo.csv')

# 2. Kredensial CDSE dari file .env
if os.path.exists(env_file):
    with open(env_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                os.environ[k.strip()] = v.strip(' "\'')

client_id = os.getenv('CDSE_CLIENT_ID')
client_secret = os.getenv('CDSE_CLIENT_SECRET')

# 3. Koordinat Poligon GeoJSON Desa Mendenrejo
geojson_mendenrejo = {
  'type': 'FeatureCollection',
  'features': [
    {
      'type': 'Feature',
      'properties': {},
      'geometry': {
        'type': 'Polygon',
        'coordinates': [
          [
            [111.4164316, -7.2300631],
            [111.474757, -7.2357873],
            [111.4730391, -7.2722541],
            [111.4116282, -7.2641575],
            [111.4164316, -7.2300631]
          ]
        ]
      }
    }
  ]
}

coords = geojson_mendenrejo['features'][0]['geometry']['coordinates'][0]
lons = [c[0] for c in coords]
lats = [c[1] for c in coords]

spatial_extent = {
    'west': min(lons),
    'east': max(lons),
    'south': min(lats),
    'north': max(lats),
    'crs': 4326
}

# 4. Rentang Waktu 1 Tahun Penuh (1 September 2025 s/d 31 Agustus 2026)
start_date = '2025-09-01'
end_date = '2026-08-31'

# 5. Inisialisasi Kerangka DataFrame CSV jika belum ada
if not os.path.exists(csv_file):
    date_range = pd.date_range(start=start_date, end=end_date, freq='D')
    df_init = pd.DataFrame({'DATE_TIME': date_range.strftime('%Y-%m-%d')})
    df_init.to_csv(csv_file, index=False)
    print(f"File CSV kerangka awal dibuat: {csv_file}")
else:
    df_init = pd.read_csv(csv_file)
    print(f"File CSV dimuat dari: {csv_file}")

# 6. Fungsi Parsing Hasil Timeseries openEO ke DataFrame
def parse_openeo_results(res_data, band_name):
    records = []
    if isinstance(res_data, dict):
        for dt_str, val_matrix in res_data.items():
            dt = dt_str[:10]
            val = np.nan
            if val_matrix is not None and len(val_matrix) > 0:
                if isinstance(val_matrix[0], list) and len(val_matrix[0]) > 0:
                    v = val_matrix[0][0]
                    val = float(v) if v is not None else np.nan
                elif not isinstance(val_matrix[0], list):
                    v = val_matrix[0]
                    val = float(v) if v is not None else np.nan
            records.append({'DATE_TIME': dt, band_name: val})
    elif isinstance(res_data, list):
        for item in res_data:
            if isinstance(item, dict):
                dt = str(item.get('date', item.get('DATE_TIME', '')))[:10]
                v = item.get(band_name, np.nan)
                records.append({'DATE_TIME': dt, band_name: float(v) if v is not None else np.nan})
    return pd.DataFrame(records)

# 7. Autentikasi Koneksi ke openEO Copernicus CDSE
logging.getLogger('openeo').setLevel(logging.ERROR)
print("\nMenghubungkan ke Copernicus Data Space Ecosystem (CDSE)...")
connection = openeo.connect('https://openeo.dataspace.copernicus.eu')

if client_id and client_secret:
    connection.authenticate_oidc_client_credentials(client_id=client_id, client_secret=client_secret)
    print("Status: BERHASIL TERAUTENTIKASI via OIDC Client Credentials!")
else:
    print("Kredensial .env belum ditemukan. Membuka login autentikasi browser...")
    connection.authenticate_oidc()

print(f"Versi API openEO Backend: {connection.capabilities().api_version()}")
print(f"File Dataset Target      : {csv_file}")
print(f"Kolom saat ini           : {df_init.columns.tolist()}")
print(f"Total baris tanggal      : {len(df_init)} hari")


File CSV kerangka awal dibuat: d:\university\semester 5_book\proyek sain data\src\2-pengambilan_data\data\data_kualitas_udara_mendenrejo.csv

Menghubungkan ke Copernicus Data Space Ecosystem (CDSE)...
Status: BERHASIL TERAUTENTIKASI via OIDC Client Credentials!
Versi API openEO Backend: 1.2.0
File Dataset Target      : d:\university\semester 5_book\proyek sain data\src\2-pengambilan_data\data\data_kualitas_udara_mendenrejo.csv
Kolom saat ini           : ['DATE_TIME']
Total baris tanggal      : 365 hari


--- 
### Sel 2: Pengambilan Data Satelit Asli — Karbon Monoksida (`CO`)

Mengambil data satelit Sentinel-5P L2 untuk band `CO` (satuan $\text{mol/m}^2$), menghitung rata-rata spasial area poligon Mendenrejo, dan menyimpannya ke kolom `CO` pada file CSV.

In [4]:
# =============================================================================
# PENGAMBILAN DATA SATELIT: CO (CARBON MONOXIDE)
# =============================================================================
print("[1/4] Mengunduh data satelit Sentinel-5P untuk band CO...")
df_main = pd.read_csv(csv_file)

# 1. Load koleksi Sentinel-5P L2 untuk band CO
cube_co = connection.load_collection(
    'SENTINEL_5P_L2',
    spatial_extent=spatial_extent,
    temporal_extent=[start_date, end_date],
    bands=['CO']
)

# 2. Agregasi rata-rata spasial poligon Desa Mendenrejo
timeseries_co = cube_co.aggregate_spatial(geometries=geojson_mendenrejo, reducer='mean')

print("Mengirim request kalkulasi spasial ke cloud CDSE openEO...")
res_co = timeseries_co.execute()

# 3. Parsing hasil riil satelit & gabungkan ke dataset
df_co = parse_openeo_results(res_co, 'CO')
df_main = df_main.drop(columns=['CO'], errors='ignore')
df_main = df_main.merge(df_co, on='DATE_TIME', how='left')

# 4. Penyimpanan ke 1 file CSV tunggal
df_main.to_csv(csv_file, index=False)
print(f"\nData CO berhasil diunduh dan disimpan ke: '{csv_file}'")
print(f"Jumlah data valid CO  : {df_main['CO'].notna().sum()} hari")
print(f"Tutupan awan / kosong : {df_main['CO'].isna().sum()} hari")
print(f"Nilai min (mol/m2)    : {df_main['CO'].min():.6f}")
print(f"Nilai max (mol/m2)    : {df_main['CO'].max():.6f}")
print(f"Nilai mean (mol/m2)   : {df_main['CO'].mean():.6f}")
print("\n5 Baris Terkini Dataset:")
print(df_main[['DATE_TIME', 'CO']].head(5))


[1/4] Mengunduh data satelit Sentinel-5P untuk band CO...
Mengirim request kalkulasi spasial ke cloud CDSE openEO...

Data CO berhasil diunduh dan disimpan ke: 'd:\university\semester 5_book\proyek sain data\src\2-pengambilan_data\data\data_kualitas_udara_mendenrejo.csv'
Jumlah data valid CO  : 215 hari
Tutupan awan / kosong : 150 hari
Nilai min (mol/m2)    : 0.020117
Nilai max (mol/m2)    : 0.040124
Nilai mean (mol/m2)   : 0.029916

5 Baris Terkini Dataset:
    DATE_TIME        CO
0  2025-09-01  0.034333
1  2025-09-02  0.021223
2  2025-09-03  0.027621
3  2025-09-04  0.027021
4  2025-09-05       NaN


--- 
### Sel 3: Pengambilan Data Satelit Asli — Nitrogen Dioksida (`NO2`)

Mengambil data asli satelit Sentinel-5P L2 untuk band `NO2` (satuan $\text{mol/m}^2$) dan menyimpannya ke kolom `NO2` pada file CSV yang sama.

In [5]:
# =============================================================================
# PENGAMBILAN DATA SATELIT ASLI: NO2 (NITROGEN DIOXIDE)
# =============================================================================
print("[2/4] Mengunduh data asli satelit Sentinel-5P untuk band NO2...")
df_main = pd.read_csv(csv_file)

# 1. Load koleksi Sentinel-5P L2 untuk band NO2
cube_no2 = connection.load_collection(
    'SENTINEL_5P_L2',
    spatial_extent=spatial_extent,
    temporal_extent=[start_date, end_date],
    bands=['NO2']
)

# 2. Agregasi rata-rata spasial poligon Desa Mendenrejo
timeseries_no2 = cube_no2.aggregate_spatial(geometries=geojson_mendenrejo, reducer='mean')

print("Mengirim request kalkulasi spasial ke cloud CDSE openEO...")
res_no2 = timeseries_no2.execute()

# 3. Parsing hasil riil satelit & gabungkan ke dataset
df_no2 = parse_openeo_results(res_no2, 'NO2')
df_main = df_main.drop(columns=['NO2'], errors='ignore')
df_main = df_main.merge(df_no2, on='DATE_TIME', how='left')

# 4. Simpan pembaruan ke 1 file CSV tunggal
df_main.to_csv(csv_file, index=False)
print(f"\nData asli NO2 berhasil diunduh dan disimpan ke: '{csv_file}'")
print(f"Jumlah data valid NO2 : {df_main['NO2'].notna().sum()} hari")
print(f"Tutupan awan / kosong : {df_main['NO2'].isna().sum()} hari")
print(f"Nilai min (mol/m2)    : {df_main['NO2'].min():.8f}")
print(f"Nilai max (mol/m2)    : {df_main['NO2'].max():.8f}")
print(f"Nilai mean (mol/m2)   : {df_main['NO2'].mean():.8f}")
print("\n5 Baris Terkini Dataset:")
print(df_main[['DATE_TIME', 'CO', 'NO2']].head(5))


[2/4] Mengunduh data asli satelit Sentinel-5P untuk band NO2...
Mengirim request kalkulasi spasial ke cloud CDSE openEO...

Data asli NO2 berhasil diunduh dan disimpan ke: 'd:\university\semester 5_book\proyek sain data\src\2-pengambilan_data\data\data_kualitas_udara_mendenrejo.csv'
Jumlah data valid NO2 : 166 hari
Tutupan awan / kosong : 199 hari
Nilai min (mol/m2)    : -0.00000177
Nilai max (mol/m2)    : 0.00005743
Nilai mean (mol/m2)   : 0.00002627

5 Baris Terkini Dataset:
    DATE_TIME        CO       NO2
0  2025-09-01  0.034333  0.000041
1  2025-09-02  0.021223  0.000042
2  2025-09-03  0.027621  0.000044
3  2025-09-04  0.027021  0.000030
4  2025-09-05       NaN  0.000019


--- 
### Sel 4: Pengambilan Data Satelit Asli — Sulfur Dioksida (`SO2`)

Mengambil data asli satelit Sentinel-5P L2 untuk band `SO2` (satuan $\text{mol/m}^2$) dan menyimpannya ke kolom `SO2` pada file CSV yang sama.

In [6]:
# =============================================================================
# PENGAMBILAN DATA SATELIT ASLI: SO2 (SULFUR DIOXIDE)
# =============================================================================
print("[3/4] Mengunduh data asli satelit Sentinel-5P untuk band SO2...")
df_main = pd.read_csv(csv_file)

# 1. Load koleksi Sentinel-5P L2 untuk band SO2
cube_so2 = connection.load_collection(
    'SENTINEL_5P_L2',
    spatial_extent=spatial_extent,
    temporal_extent=[start_date, end_date],
    bands=['SO2']
)

# 2. Agregasi rata-rata spasial poligon Desa Mendenrejo
timeseries_so2 = cube_so2.aggregate_spatial(geometries=geojson_mendenrejo, reducer='mean')

print("Mengirim request kalkulasi spasial ke cloud CDSE openEO...")
res_so2 = timeseries_so2.execute()

# 3. Parsing hasil riil satelit & gabungkan ke dataset
df_so2 = parse_openeo_results(res_so2, 'SO2')
df_main = df_main.drop(columns=['SO2'], errors='ignore')
df_main = df_main.merge(df_so2, on='DATE_TIME', how='left')

# 4. Simpan pembaruan ke 1 file CSV tunggal
df_main.to_csv(csv_file, index=False)
print(f"\nData asli SO2 berhasil diunduh dan disimpan ke: '{csv_file}'")
print(f"Jumlah data valid SO2 : {df_main['SO2'].notna().sum()} hari")
print(f"Tutupan awan / kosong : {df_main['SO2'].isna().sum()} hari")
print(f"Nilai min (mol/m2)    : {df_main['SO2'].min():.8f}")
print(f"Nilai max (mol/m2)    : {df_main['SO2'].max():.8f}")
print(f"Nilai mean (mol/m2)   : {df_main['SO2'].mean():.8f}")
print("\n5 Baris Terkini Dataset:")
print(df_main[['DATE_TIME', 'CO', 'NO2', 'SO2']].head(5))


[3/4] Mengunduh data asli satelit Sentinel-5P untuk band SO2...
Mengirim request kalkulasi spasial ke cloud CDSE openEO...

Data asli SO2 berhasil diunduh dan disimpan ke: 'd:\university\semester 5_book\proyek sain data\src\2-pengambilan_data\data\data_kualitas_udara_mendenrejo.csv'
Jumlah data valid SO2 : 224 hari
Tutupan awan / kosong : 141 hari
Nilai min (mol/m2)    : -0.00063455
Nilai max (mol/m2)    : 0.00099346
Nilai mean (mol/m2)   : 0.00007636

5 Baris Terkini Dataset:
    DATE_TIME        CO       NO2       SO2
0  2025-09-01  0.034333  0.000041  0.000342
1  2025-09-02  0.021223  0.000042 -0.000061
2  2025-09-03  0.027621  0.000044  0.000040
3  2025-09-04  0.027021  0.000030  0.000272
4  2025-09-05       NaN  0.000019 -0.000481


--- 
### Sel 5: Pengambilan Data Satelit Asli — Metana (`CH4`)

Mengambil data asli satelit Sentinel-5P L2 untuk band `CH4` dan melengkapi seluruh 4 polutan pada dataset CSV tunggal.

In [7]:
# =============================================================================
# PENGAMBILAN DATA SATELIT ASLI: CH4 (METHANE)
# =============================================================================
print("[4/4] Mengunduh data asli satelit Sentinel-5P untuk band CH4...")
df_main = pd.read_csv(csv_file)

# 1. Load koleksi Sentinel-5P L2 untuk band CH4
cube_ch4 = connection.load_collection(
    'SENTINEL_5P_L2',
    spatial_extent=spatial_extent,
    temporal_extent=[start_date, end_date],
    bands=['CH4']
)

# 2. Agregasi rata-rata spasial poligon Desa Mendenrejo
timeseries_ch4 = cube_ch4.aggregate_spatial(geometries=geojson_mendenrejo, reducer='mean')

print("Mengirim request kalkulasi spasial ke cloud CDSE openEO...")
res_ch4 = timeseries_ch4.execute()

# 3. Parsing hasil riil satelit & gabungkan ke dataset
df_ch4 = parse_openeo_results(res_ch4, 'CH4')
df_main = df_main.drop(columns=['CH4'], errors='ignore')
df_main = df_main.merge(df_ch4, on='DATE_TIME', how='left')

# 4. Simpan pembaruan ke 1 file CSV tunggal
df_main.to_csv(csv_file, index=False)
print(f"\nData asli CH4 berhasil diunduh dan disimpan ke: '{csv_file}'")
print(f"Jumlah data valid CH4 : {df_main['CH4'].notna().sum()} hari")
print(f"Tutupan awan / kosong : {df_main['CH4'].isna().sum()} hari")
if df_main['CH4'].notna().any():
    print(f"Nilai min             : {df_main['CH4'].min():.4f}")
    print(f"Nilai max             : {df_main['CH4'].max():.4f}")
    print(f"Nilai mean            : {df_main['CH4'].mean():.4f}")
print("\n5 Baris Terkini Dataset:")
print(df_main.head(5))


[4/4] Mengunduh data asli satelit Sentinel-5P untuk band CH4...
Mengirim request kalkulasi spasial ke cloud CDSE openEO...


ReadTimeout: HTTPSConnectionPool(host='openeo.dataspace.copernicus.eu', port=443): Read timed out. (read timeout=1800)

--- 
### Sel 6: Audit Akhir Dataset Tunggal 4 Polutan Asli

Memeriksa kelengkapan, dimensi, ringkasan statistik, dan pratinjau seluruh 4 polutan asli yang telah tersimpan dalam 1 file CSV terpadu.

In [ ]:
# =============================================================================
# AUDIT FINAL DATASET TUNGGAL
# =============================================================================
df_final = pd.read_csv(csv_file)

print("="*85)
print("AUDIT FINAL DATASET ASLI SATELIT SENTINEL-5P DESA MENDENREJO:")
print("="*85)
print(f"Lokasi File Dataset : {csv_file}")
print(f"Total Dimensi Data  : {df_final.shape[0]} baris x {df_final.shape[1]} kolom")
print(f"Daftar Kolom        : {df_final.columns.tolist()}")
print(f"Periode Penanggalan : {df_final['DATE_TIME'].iloc[0]} s/d {df_final['DATE_TIME'].iloc[-1]} ({len(df_final)} hari)")
print("\nAudit Nilai Kosong (Tutupan Awan / Missing Values):")
print(df_final.isnull().sum())

print("\nStatistik Deskriptif 4 Polutan Asli Satelit:")
print(df_final.describe())

print("\n10 Baris Pertama Dataset:")
print(df_final.head(10))
print("="*85)
print("Status: SELURUH 4 POLUTAN SATELIT ASLI TELAH LENGKAP TERSIMPAN DALAM 1 CSV!")
